<a href="https://colab.research.google.com/github/gline4u-gif/main.py/blob/main/Stratagy_Backtesting__Tradingview4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CELL 1: The Installation Script

In [ ]:

# ==========================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# Run this cell ONLY ONCE when you open Colab
# ==========================================
print("📦 Installing required libraries...")

!pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

!pip install --upgrade tvDatafeed
!pip install gspread
!pip install gspread-dataframe
!pip install pandas



print("✅ Installation complete! You can now run the Engine in Cell 2.")

📦 Installing required libraries...
  Cloning https://github.com/rongardF/tvdatafeed.git to /tmp/pip-req-build-ov5d7kqk
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed.git /tmp/pip-req-build-ov5d7kqk
  Resolved https://github.com/rongardF/tvdatafeed.git to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done
✅ Installation complete! You can now run the Engine in Cell 2.


🚀 CELL 2: The Master Backtesting Engine

In [ ]:

# ==========================================
# CELL 2: THE MASTER TRADING ENGINE
# ==========================================
import pandas as pd
import time
import random
from tvDatafeed import TvDatafeed, Interval
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe
from google.auth import default

def run_ultimate_trading_terminal():
    # ------------------------------------------
    # 1. READ GOOGLE SHEET CONTROL PANEL
    # ------------------------------------------
    print("1. 🔑 Authenticating Google Workspace...")
    try:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
    except Exception as e:
        print(f"❌ Auth Error: {e}")
        return

    SHEET_ID = "1Qzp9s3GdRLfW2J81bdJ2PvJYAOIcdPk_JBMLrZjTr8Q"
    TAB_NAME = "Stratagy_Backtesting _Tradingview"

    print(f"2. 🛰️ Reading Control Panel...")
    try:
        spreadsheet = gc.open_by_key(SHEET_ID)
        worksheet = spreadsheet.worksheet(TAB_NAME)

        interval_input = worksheet.acell('B1').value
        start_date_str = worksheet.acell('B2').value
        end_date_str = worksheet.acell('B3').value

        raw_symbols = [
            worksheet.acell('C2').value,
            worksheet.acell('C3').value,
            worksheet.acell('C4').value,
            worksheet.acell('C5').value
        ]

        target_symbols = [str(s).strip() for s in raw_symbols if s and str(s).strip() != '']

        if not target_symbols:
            print("❌ No symbols found! Please enter at least one symbol in C2-C5.")
            return

        interval_input = str(interval_input).strip().upper() if interval_input else '15'

        print(f"   -> Found {len(target_symbols)} Legs.")
        print(f"   -> Timeframe: {interval_input} Minute(s)")

    except Exception as e:
        print(f"❌ Error reading Google Sheet: {e}")
        return

    # ------------------------------------------
    # 2. THE INTERVAL TRANSLATOR
    # ------------------------------------------
    interval_map = {
        '1': Interval.in_1_minute, '3': Interval.in_3_minute, '5': Interval.in_5_minute,
        '15': Interval.in_15_minute, '30': Interval.in_30_minute, '45': Interval.in_45_minute,
        '60': Interval.in_1_hour, 'D': Interval.in_daily, 'W': Interval.in_weekly
    }
    tv_interval = interval_map.get(interval_input)

    # ------------------------------------------
    # 3. FETCH EACH LEG (WITH JITTER)
    # ------------------------------------------
    tv = TvDatafeed()
    all_leg_data = []

    for idx, symbol in enumerate(target_symbols):
        print(f"\n   🔌 [Leg {idx+1}] Fetching data for {symbol}...")
        try:
            df = tv.get_hist(symbol=symbol, exchange='NSE', interval=tv_interval, n_bars=5000)

            # 🕵️ JITTER: Sleep for a random time to avoid API bans
            micro_sleep = random.uniform(1.5, 4.2)
            time.sleep(micro_sleep)

        except Exception as e:
            print(f"   ❌ Fetch Error for {symbol}: {e}")
            continue

        if df is None or df.empty:
            print(f"   ⚠️ No data found for {symbol}. Skipping.")
            continue

        df.reset_index(inplace=True)
        df = df[['datetime', 'open', 'high', 'low', 'close', 'volume']]

        df['datetime'] = pd.to_datetime(df['datetime'])
        df['datetime'] = df['datetime'].dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')
        df['datetime'] = df['datetime'].dt.tz_localize(None)

        if start_date_str and str(start_date_str).strip() != 'None':
            try:
                start_dt = pd.to_datetime(start_date_str, dayfirst=True)
                df = df[df['datetime'] >= start_dt]
            except: pass

        if end_date_str and str(end_date_str).strip() != 'None':
            try:
                end_dt = pd.to_datetime(end_date_str, dayfirst=True)
                if end_dt.hour == 0 and end_dt.minute == 0 and end_dt.second == 0:
                    end_dt = end_dt.replace(hour=23, minute=59, second=59)
                df = df[df['datetime'] <= end_dt]
            except: pass

        df.insert(0, 'Symbol', symbol)
        all_leg_data.append(df)
        print(f"   ✅ Leg {idx+1} Raw Download: {len(df)} candles.")

    if len(all_leg_data) != len(target_symbols):
        print("\n❌ Could not fetch data for all legs. Please check the symbols.")
        return

    # ------------------------------------------
    # 4. THE STRICT MARKET SYNCHRONIZER
    # ------------------------------------------
    print("\n4. ⚙️ Synchronizing timeframes and applying strict market hours...")

    common_start = max([df['datetime'].min() for df in all_leg_data])

    # 🚨 MAX FIX: Prevents missing morning candles!
    common_end = max([df['datetime'].max() for df in all_leg_data])

    if interval_input in ['1', '3', '5', '15', '30', '45', '60']:
        freq_str = f'{interval_input}min'
    else:
        freq_str = 'D'

    master_timeline = pd.date_range(start=common_start, end=common_end, freq=freq_str)
    aligned_legs = []

    for idx, df in enumerate(all_leg_data):
        df.set_index('datetime', inplace=True)

        aligned_df = df.reindex(master_timeline)
        aligned_df['close'] = aligned_df['close'].ffill().bfill()
        aligned_df['open'] = aligned_df['open'].ffill().bfill()
        aligned_df['high'] = aligned_df['high'].ffill().bfill()
        aligned_df['low'] = aligned_df['low'].ffill().bfill()
        aligned_df['volume'] = aligned_df['volume'].fillna(0)
        aligned_df['Symbol'] = target_symbols[idx]

        # 🚨 STRICT MARKET HOURS (9:15 to 3:30)
        aligned_df = aligned_df.between_time('09:15', '15:30')

        aligned_df.index.name = 'datetime'
        aligned_df.reset_index(inplace=True)

        # 🚨 REMOVE WEEKENDS
        aligned_df = aligned_df[aligned_df['datetime'].dt.dayofweek < 5]

        aligned_df = aligned_df.sort_values(by='datetime', ascending=False)
        aligned_df['datetime'] = aligned_df['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')

        aligned_legs.append(aligned_df)

    print(f"   ✅ Market boundaries enforced! Purged all night & weekend candles.")

    # ------------------------------------------
    # 5. UPLOAD TO GOOGLE SHEETS
    # ------------------------------------------
    try:
        print(f"\n5. 🧹 Purging old data in Safe Zone...")
        # 🚨 FORMULA WALL: Only clears F through AJ. Leaves AL+ completely safe.
        worksheet.batch_clear(['F6:AJ5000'])

        print(f"6. 📤 Syncing Locked Data to '{TAB_NAME}'...")
        # 🚨 OFFSET LAYOUT: Starts on Column 6 (F), Row 6
        starting_columns = [6, 14, 22, 30]

        for idx, df in enumerate(aligned_legs):
            target_col = starting_columns[idx]
            set_with_dataframe(worksheet, df, row=6, col=target_col, include_index=False, include_column_header=True)
            print(f"   -> Uploaded strict intraday Leg {idx+1} to Column {chr(64+target_col) if target_col <= 26 else chr(64+target_col//26) + chr(64+target_col%26)}")

        print("\n🎉 SUCCESS! Data updated.")

    except Exception as e:
        print(f"\n❌ Upload Error: {e}")

# ------------------------------------------
# 6. THE RANDOMIZED AUTO-PILOT LOOP
# ------------------------------------------
print("🚀 Starting Randomized Auto-Updater...")

while True:
    run_ultimate_trading_terminal()

    # 🕵️ JITTER: Pick a random wait time between 10 and 25 seconds
    wait_time = random.randint(10, 25)

    print(f"\n⏳ Update complete! Mimicking human behavior... sleeping for {wait_time} seconds...")
    print("⚠️ (Leave this browser tab open to keep the bot running)")

    time.sleep(wait_time)

🚀 Starting Randomized Auto-Updater...
1. 🔑 Authenticating Google Workspace...
2. 🛰️ Reading Control Panel...


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 25 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 25 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 11 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 16 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


...

   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 19 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 22 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 23 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 11 seconds...
⚠️ (Leave this browser tab open to keep the bot running)


   -> Found 4 Legs.
   -> Timeframe: 15 Minute(s)

   🔌 [Leg 1] Fetching data for TCS260428P2440...
   ✅ Leg 1 Raw Download: 351 candles.

   🔌 [Leg 2] Fetching data for TCS260428P2460...
   ✅ Leg 2 Raw Download: 313 candles.

   🔌 [Leg 3] Fetching data for TCS260428C2620...
   ✅ Leg 3 Raw Download: 287 candles.

   🔌 [Leg 4] Fetching data for TCS260428C2640...
   ✅ Leg 4 Raw Download: 378 candles.

4. ⚙️ Synchronizing timeframes and applying strict market hours...
   ✅ Market boundaries enforced! Purged all night & weekend candles.

5. 🧹 Purging old data in Safe Zone...
6. 📤 Syncing Locked Data to 'Stratagy_Backtesting _Tradingview'...
   -> Uploaded strict intraday Leg 1 to Column F
   -> Uploaded strict intraday Leg 2 to Column N
   -> Uploaded strict intraday Leg 3 to Column V
   -> Uploaded strict intraday Leg 4 to Column AD

🎉 SUCCESS! Data updated.

⏳ Update complete! Mimicking human behavior... sleeping for 20 seconds...
⚠️ (Leave this browser tab open to keep the bot running)
